# Entregável 6 — Extração de Atributos do ECG (Feature Extraction)

**Disciplina:** Aquisição e Processamento de Biossinais  
**Equipe:** José Ferreira Lessa & Matheus Rocha Gomes da Silva  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL — A Large Publicly Available Electrocardiography Dataset (PhysioNet)  
**Referência:** Wagner et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.  
**Data:** Abril e Maio de 2026

---

## Objetivo

Este notebook descreve a etapa de **Extração de Atributos (Feature Extraction)**, responsável por transformar os sinais de ECG — já limpos, segmentados e organizados nos Entregáveis anteriores — em um conjunto estruturado de variáveis numéricas que representam a atividade cardíaca de forma compacta e informativa.

A extração de features é, do ponto de vista metodológico, a etapa central do pipeline de Reconhecimento de Padrões: é aqui que o conhecimento clínico da cardiologia é traduzido em linguagem matemática. Um sinal de ECG bruto não pode ser passado diretamente a um classificador — é necessário destilar a informação relevante em atributos bem definidos, cada um com interpretação fisiológica clara.

A estratégia adotada neste trabalho, justificada no Entregável 5, é **híbrida e em dois níveis**:

1. **Nível Global (registro de 10 s):** capturamos a dinâmica geral do exame em todas as 12 derivações, extraindo estatísticas de energia, conteúdo espectral e descritores de forma.
2. **Nível Local (batimento individual):** analisamos a morfologia fina de cada ciclo cardíaco detectado no Entregável 5, medindo amplitudes de ondas, durações de segmentos e métricas de variabilidade entre batimentos (HRV). Os valores individuais são então **agregados** (mediana e desvio padrão) para o nível do registro, mantendo a consistência da instância principal.

Essa arquitetura permite representar tanto **alterações de ritmo** (capturadas via HRV) quanto **alterações morfológicas** — elevação de ST, hipertrofia ventricular, bloqueios de ramo — de forma robusta e sem replicar artificialmente os labels diagnósticos.

O notebook está organizado nas seguintes seções:

1. **Fundamentação Teórica e Taxonomia dos Atributos:** revisão dos domínios de extração (tempo, frequência, tempo-frequência e não-linear) e justificativa para a seleção de cada grupo de features adotado neste trabalho.
2. **Importações, Configurações e Carregamento:** setup do ambiente, carregamento dos sinais limpos (Entregável 4) e das segmentações por batimento (Entregável 5).
3. **Domínio do Tempo — Estatísticas Globais (12 derivações):** extração de descritores de energia e forma sobre o registro completo de 10 s.
4. **Domínio da Frequência — PSD via Welch (12 derivações):** análise do conteúdo espectral por bandas fisiológicas e descritores de forma da PSD.
5. **Morfologia e Variabilidade de Ritmo (HRV):** métricas morfológicas extraídas por batimento e agregadas ao nível do registro; métricas temporais de HRV sobre a série RR.
6. **Domínio Tempo-Frequência — Wavelet Discreta (DWT):** decomposição por subbandas com a família Daubechies db4 e extração de energia e entropia por nível.
7. **Dinâmica Não-Linear:** dimensão fractal de Higuchi, análise DFA e métricas do Diagrama de Poincaré.
8. **Consolidação e Geração do Dataset Final (features_raw):** unificação de todos os domínios em um único DataFrame, com salvamento em Parquet.
9. **Visualização e Sanity Check:** verificações qualitativas e quantitativas da consistência do dataset gerado.
10. **Síntese e Conexão com o Entregável 7.**

> **Nota metodológica:** Todas as features são extraídas de forma **independente por fold**, respeitando a separação entre treino, validação e teste definida no Entregável 1. Nenhum parâmetro aprendido (ex: limiares adaptativos) é estimado sobre o conjunto de validação ou teste, evitando *data leakage* desde esta etapa.

---

## Seção 1 — Fundamentação Teórica e Taxonomia dos Atributos

### 1.1 Por que extrair atributos manualmente?

Em abordagens de *Deep Learning* com CNN ou Transformers, o modelo aprende representações internas diretamente do sinal bruto. No entanto, para o pipeline de ML clássico (SVM, kNN, Random Forest) que será aplicado no componente de Reconhecimento de Padrões desta disciplina, é necessário fornecer ao modelo uma **representação vetorial** de dimensão fixa para cada instância.

A extração manual de features tem três vantagens concretas em relação ao uso de sinais brutos:

- **Interpretabilidade:** cada feature tem significado clínico ou físico bem definido — é possível explicar ao médico por que determinada feature contribuiu para uma classificação.
- **Robustez a variações de escala:** features normalizadas (ex: frequência de pico, razão de bandas) são menos sensíveis a diferenças de impedância eletrodo-pele entre pacientes.
- **Eficiência computacional:** um vetor de ~200 features por registro é ordens de grandeza menor que o sinal bruto de 1000×12 amostras, tornando o treinamento mais rápido e menos sujeito à maldição da dimensionalidade.

---

### 1.2 Taxonomia dos Domínios de Extração

A literatura de processamento de biossinais organiza os atributos de ECG em quatro domínios principais, que se complementam e capturam aspectos distintos da atividade cardíaca:

---

#### A. Domínio do Tempo

Opera diretamente sobre as amplitudes do sinal sem qualquer transformação de domínio. São os atributos mais simples computacionalmente, mas altamente informativos para caracterizar a **distribuição de energia e a forma estatística** do sinal.

Atributos típicos:
- **RMS (Root Mean Square):** medida de energia média do sinal. Relacionado à amplitude típica das ondas. Elevado em sinais com alta amplitude (ex: hipertrofia ventricular).
- **MAV (Mean Absolute Value):** análogo ao RMS, porém sem quadratura. Menos sensível a picos extremos.
- **Variância:** dispersão das amplitudes. Alta variância pode indicar ritmo irregular ou presença de artefatos residuais.
- **Peak-to-Peak (p2p):** diferença entre máximo e mínimo. Sensível à amplitude das deflexões.
- **Zero Crossing Rate (ZCR):** frequência com que o sinal cruza o zero. Correlaciona-se com o conteúdo de alta frequência do sinal.
- **Skewness e Kurtosis:** assimetria e curtose da distribuição de amplitudes. Capturam morfologias assimétricas (ex: onda R muito maior que onda S).

---

#### B. Domínio da Frequência

Transforma o sinal do domínio temporal para o domínio espectral, permitindo identificar quais componentes de frequência concentram a energia do sinal.

No ECG, as estruturas morfológicas têm faixas de frequência características:
- **Onda P:** 0,5 – 5 Hz
- **Complexo QRS:** 5 – 25 Hz (com energia principal acima de 10 Hz)
- **Onda T:** 0,5 – 5 Hz

A **Densidade Espectral de Potência (PSD)** estimada via **método de Welch** é preferível à FFT simples por ser um estimador estatisticamente mais robusto: divide o sinal em segmentos sobrepostos, calcula o periodograma de cada um e faz a média, reduzindo a variância da estimativa espectral.

---

#### C. Domínio Tempo-Frequência

Busca capturar como o **conteúdo espectral varia ao longo do tempo** — algo que a PSD global não consegue representar, pois assume estacionaridade do sinal.

A **Transformada Wavelet Discreta (DWT)** decompõe o sinal em subbandas de frequência através de filtros passa-alta (coeficientes de detalhe, D) e passa-baixa (coeficientes de aproximação, A), aplicados de forma iterativa e com subamostragem (*downsampling*) a cada nível.

A família **Daubechies db4** é amplamente utilizada em ECG por apresentar boa localização temporal e formato de wavelet-mãe compatível com as morfologias das ondas cardíacas.

A energia e a entropia de cada subbanda refletem, respectivamente, o **volume** e a **complexidade distribucional** da atividade em cada faixa de frequência.

---

#### D. Dinâmica Não-Linear

O coração não é um sistema puramente linear — sua regulação envolve interações complexas entre o sistema nervoso autônomo simpático e parassimpático, o nó sinusal e o sistema His-Purkinje. Essa complexidade se manifesta em padrões fractais e caóticos na série temporal de intervalos RR e no próprio sinal de ECG.

Atributos não-lineares capturam essa complexidade de formas que os domínios anteriores não conseguem:

- **Dimensão Fractal de Higuchi (FD):** mede a complexidade geométrica do sinal. Sinais mais irregulares têm FD maior. Redução da FD pode indicar perda de complexidade associada a patologias.
- **DFA (Detrended Fluctuation Analysis):** quantifica as correlações de longo alcance no sinal, estimando o expoente de escala α. α ≈ 1,0 indica dinâmica fractal saudável; α < 0,5 indica anti-correlação (típica de arritmias graves).
- **Entropia de Amostra (SampEn):** mede a imprevisibilidade do sinal — quão difícil é predizer o próximo valor a partir dos anteriores. Coração saudável tende a ter SampEn intermediária; patologias frequentemente reduzem a entropia (ritmo mais regular/mecânico) ou a elevam (ritmo caótico).
- **Métricas do Diagrama de Poincaré (SD1, SD2):** representação geométrica da variabilidade do intervalo RR. SD1 captura variabilidade de curto prazo (atividade parassimpática); SD2 captura variabilidade de longo prazo (atividade simpática + parassimpática).

---

### 1.3 Síntese da Estratégia de Extração

A tabela abaixo resume os grupos de features extraídos neste entregável, sua base de cálculo e o nível de análise correspondente:

| Grupo | Domínio | Base de Cálculo | Nível | Nº Derivações |
|---|---|---|---|---|
| Estatísticas Temporais | Tempo | Registro 10s | Global | 12 |
| Potência Espectral (Welch) | Frequência | Registro 10s | Global | 12 |
| Morfologia de Ondas | Tempo | Batimento (agregado) | Local → Global | DII e V5 |
| HRV Temporal | Tempo | Série RR | Local → Global | — |
| Subbandas Wavelet (DWT) | Tempo-Frequência | Registro 10s | Global | DII |
| Dimensão Fractal / DFA | Não-Linear | Registro 10s | Global | DII |
| SampEn / Poincaré (SD1, SD2) | Não-Linear | Série RR | Local → Global | — |

> **Nota:** Para os domínios mais custosos computacionalmente (Wavelet, DFA, SampEn), o cálculo é realizado sobre a **derivação DII**, que é a derivação de referência clínica para análise de ritmo e morfologia geral do ECG. Para os domínios menos custosos (tempo e frequência), o cálculo é feito para **todas as 12 derivações**, maximizando a cobertura diagnóstica sem comprometer a viabilidade computacional.

---

## 2. Importações, Configurações e Dependências

Todas as bibliotecas utilizadas neste notebook já foram apresentadas em entregáveis anteriores, com exceção de `PyWavelets` e `antropy`, descritas abaixo.

### Novas dependências adicionadas neste entregável

- **`PyWavelets` (pywt):** biblioteca para **Transformada Wavelet Discreta e Contínua** em Python. Utilizada para a decomposição do sinal ECG em subbandas de frequência via família Daubechies (db4). Mantida ativamente e amplamente utilizada em biossinal processing.

- **`antropy`:** biblioteca especializada em **métricas de complexidade e entropia** para séries temporais. Implementa de forma eficiente e numericamente estável a Entropia de Amostra (SampEn), a Dimensão Fractal de Higuchi e a Análise de Flutuações Sem Tendência (DFA). Preferida aqui por sua validação contra implementações de referência da literatura.

As demais bibliotecas (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `joblib`, `tqdm`) já foram extensamente utilizadas e descritas nos Entregáveis anteriores.

In [ ]:
import os
import ast
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.signal as signal
import scipy.stats as stats
from scipy.stats import entropy
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm
from IPython.display import display, Markdown

# Bibliotecas especializadas para análise de biossinais
# Descomente a linha abaixo caso não tenha instalado ainda
# %pip install PyWavelets antropy fastparquet pyarrow -q

import pywt
import antropy as ant

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 3. Configurações Globais

Constantes do projeto, consistentes com os Entregáveis anteriores.

In [ ]:
FS          = 100           # Frequência de amostragem (Hz)
N_LEADS     = 12            # Número de derivações
LEAD_NAMES  = ['I', 'II', 'III', 'aVL', 'aVR', 'aVF',
                'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

FOLDS_TREINO = [1, 2, 3, 4, 5, 6, 7, 8]
FOLD_VAL     = 9
FOLD_TEST    = 10

# Diretórios de entrada (produtos dos Entregáveis 4 e 5)
DIR_IN_D4 = Path('../../entregavel-4/outputs/')
DIR_IN_D5 = Path('../../entregavel-5/outputs/')

# Diretórios de saída deste entregável
FIGS_DIR = Path('../figuras/')
OUT_DIR  = Path('../outputs/')

for d in [FIGS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')
print(f'Figuras em  : {FIGS_DIR.resolve()}')
print(f'Outputs em  : {OUT_DIR.resolve()}')

---

## 4. Carregamento dos Sinais e Metadados

Partimos dos arquivos gerados ao final dos Entregáveis 4 e 5:

- `sinais_limpos_100hz.npy` (Entregável 4): array de shape `(N, 1000, 12)` com os sinais filtrados e winsorizados, prontos para extração de features globais.
- `batimentos_segmentados.npy` (Entregável 5): array de shape `(N_bat, 60, 12)` com as janelas de 600 ms centradas em cada pico R detectado.
- `registros_ids.csv` (Entregável 5): mapeamento `ecg_id → metadados` dos registros aprovados.
- `batimentos_ids.csv` (Entregável 5): mapeamento `batimento → ecg_id + rr_interval_ms`.

O carregamento dos sinais é feito via **Memory Mapping** (`mmap_mode='r'`), que mantém os dados em disco e carrega apenas as fatias necessárias em RAM durante o processamento. Para um array de sinais na casa dos GB, isso é indispensável para evitar erros de memória.

In [ ]:
npy_sinais   = DIR_IN_D4 / 'sinais_limpos_100hz.npy'
npy_bats     = DIR_IN_D5 / 'batimentos_segmentados.npy'
csv_reg_ids  = DIR_IN_D5 / 'registros_ids.csv'
csv_beat_ids = DIR_IN_D5 / 'batimentos_ids.csv'

for p in [npy_sinais, npy_bats, csv_reg_ids, csv_beat_ids]:
    if not p.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {p}\n"
            f"Execute os Entregáveis 4 e 5 antes de prosseguir."
        )

print("Carregando base de dados via Memory Mapping...")

# mmap_mode='r' lê o arquivo sem carregar tudo na RAM de uma vez
sinais_10s  = np.load(str(npy_sinais), mmap_mode='r')
batimentos  = np.load(str(npy_bats),   mmap_mode='r')

df_reg_ids  = pd.read_csv(str(csv_reg_ids),  index_col='ecg_id')
df_beat_ids = pd.read_csv(str(csv_beat_ids))

# Recuperar superclasse (salva como string em CSV)
if 'diagnostic_superclass' in df_reg_ids.columns:
    df_reg_ids['diagnostic_superclass'] = (
        df_reg_ids['diagnostic_superclass']
        .apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    )

print(f"Sinais 10s   : {sinais_10s.shape}  (registros × amostras × derivações)")
print(f"Batimentos   : {batimentos.shape}  (batimentos × amostras × derivações)")
print(f"Registros    : {len(df_reg_ids)}")
print(f"Batimentos ID: {len(df_beat_ids)} linhas")

# Integridade: índices devem coincidir
assert sinais_10s.shape[0] == len(df_reg_ids), (
    f"Divergência: {sinais_10s.shape[0]} sinais vs {len(df_reg_ids)} linhas nos metadados"
)
print("\nIntegridade verificada — índices alinhados.")

---

## Seção 2 — Domínio do Tempo: Estatísticas Globais (12 Derivações)

### 2.1 Fundamentação

Os atributos temporais são calculados diretamente sobre os vetores de amplitude do sinal, sem qualquer transformação de domínio. Apesar de serem os mais simples do conjunto, capturam informação valiosa sobre a **distribuição de energia e a forma estatística** do sinal em cada derivação.

O cálculo é feito sobre o registro completo de 10 s (1000 amostras) e repetido para as 12 derivações, gerando um subconjunto de **7 features × 12 derivações = 84 features**.

Os atributos extraídos são:

- **RMS:** $\text{RMS} = \sqrt{\frac{1}{N} \sum_{n=1}^{N} x[n]^2}$ — energia quadrática média. Derivações com maior RMS tendem a ser aquelas com deflexões de maior amplitude (ex: DII em sinais normais, V4 em hipertrofias).

- **MAV:** $\text{MAV} = \frac{1}{N} \sum_{n=1}^{N} |x[n]|$ — média dos valores absolutos. Alternativa ao RMS com menor sensibilidade a picos isolados, por não elevar os valores ao quadrado.

- **Variância:** $\sigma^2 = \frac{1}{N} \sum (x[n] - \bar{x})^2$ — dispersão das amplitudes em torno da média. Alta variância pode indicar ritmo irregular ou grande modulação de amplitude entre ondas.

- **Peak-to-Peak:** diferença entre o valor máximo e o mínimo do sinal. Amplitudes p2p elevadas são características de hipertrofia ventricular nas derivações precordiais.

- **Zero Crossing Rate (ZCR):** proporção de amostras em que o sinal muda de sinal. Correlaciona-se com o conteúdo de alta frequência — sinais com mais componentes rápidas (ex: maior contribuição do QRS) tendem a ter ZCR maior.

- **Skewness:** assimetria da distribuição de amplitudes. Um ECG com onda R dominante (complexo QRS positivo) tende a apresentar skewness positiva na derivação DII.

- **Kurtosis:** curtose da distribuição. Sinais com picos muito abruptos (ex: QRS estreito e de alta amplitude) apresentam kurtosis elevada.

### 2.2 Implementação

In [ ]:
def extract_time_domain(sig_12l):
    """
    Calcula estatísticas temporais para as 12 derivações de um registro de 10s.

    Parâmetros:
    - sig_12l : array (1000, 12) com o sinal de 12 derivações

    Retorna:
    - feats : dicionário com as features no formato {nome_feature: valor}
    """
    feats = {}
    for i, lead in enumerate(LEAD_NAMES):
        s   = sig_12l[:, i].astype(np.float64)
        ser = pd.Series(s)

        feats[f'time_rms_{lead}']  = np.sqrt(np.mean(s ** 2))
        feats[f'time_mav_{lead}']  = np.mean(np.abs(s))
        feats[f'time_var_{lead}']  = np.var(s)
        feats[f'time_p2p_{lead}']  = float(np.ptp(s))

        # ZCR: proporção de amostras com mudança de sinal
        feats[f'time_zcr_{lead}']  = np.sum(np.diff(np.sign(s)) != 0) / len(s)

        # Momentos de forma da distribuição de amplitudes (via pandas)
        feats[f'time_skew_{lead}'] = float(ser.skew())
        feats[f'time_kurt_{lead}'] = float(ser.kurt())

    return feats


print("Processando Domínio do Tempo em paralelo...")
time_feats_list = Parallel(n_jobs=-1)(
    delayed(extract_time_domain)(sinais_10s[i])
    for i in tqdm(range(len(df_reg_ids)), desc='Tempo')
)

df_time = pd.DataFrame(time_feats_list, index=df_reg_ids.index)

del time_feats_list
gc.collect()

print(f"\nFeatures temporais extraídas: {df_time.shape[1]} colunas")
display(df_time.describe().T[['mean', 'std', 'min', 'max']].head(10))

### 2.3 Visualização: Distribuição do RMS por Derivação e Classe

O boxplot abaixo mostra a distribuição do RMS na derivação DII separada por superclasse diagnóstica. Esperamos observar:

- **HYP (Hipertrofia):** valores de RMS tipicamente mais elevados, pela maior massa muscular que amplifica as deflexões do ECG.
- **NORM:** distribuição mais concentrada e com menor variância.
- **MI (Infarto do Miocárdio):** possível redução de amplitude em derivações específicas pela perda de massa miocárdica viável.

[Inserir análise comparativa após execução — comentar se as diferenças entre classes são visualmente expressivas e se há sobreposição entre MI e NORM que justifique o uso de features de outros domínios para melhor discriminação.]

In [ ]:
# Preparação do DataFrame para plot
df_plot_time = df_time[['time_rms_II', 'time_rms_V5']].copy()

# Extração do label primário (primeira superclasse)
def get_primary(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return 'OUTRO'

df_plot_time['label'] = df_reg_ids['diagnostic_superclass'].apply(get_primary)
order_classes = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, lead_name in zip(axes,
                               ['time_rms_II', 'time_rms_V5'],
                               ['DII', 'V5']):
    df_tmp = df_plot_time[df_plot_time['label'].isin(order_classes)]
    sns.boxplot(data=df_tmp, x='label', y=col, order=order_classes,
                hue='label', palette='Spectral', legend=False, ax=ax)
    ax.set_title(f'Distribuição do RMS — Derivação {lead_name}', fontsize=13)
    ax.set_xlabel('Superclasse Diagnóstica')
    ax.set_ylabel('RMS (mV)')
    ax.set_yscale('log')

plt.suptitle('Energia RMS por Classe Diagnóstica (Domínio Temporal)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'time_rms_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

**📝 Comentários — Distribuição do RMS por Classe (DII e V5):**

> *Preencha esta seção após executar a célula acima. Sugestões do que analisar:*

- **HYP vs. NORM:** espera-se que a classe HYP (Hipertrofia Ventricular) apresente mediana de RMS visivelmente superior às demais, especialmente em V5, que é a derivação precordial de maior sensibilidade para a massa ventricular esquerda. Verifique se essa separação é expressiva ou se há sobreposição com NORM.
- **MI vs. NORM:** em infarto transmural, a perda de massa miocárdica viável pode reduzir a amplitude das deflexões nas derivações que "enxergam" a zona infartada. Analise se MI apresenta RMS sistematicamente menor ou se a sobreposição com NORM é grande, o que justificaria a necessidade de features espectrais e morfológicas para melhor discriminação.
- **Escalas e outliers:** o eixo y em escala logarítmica facilita a comparação entre classes com amplitudes muito diferentes. Comente se há outliers expressivos que possam indicar registros com artefatos residuais não removidos no Entregável 4.
- **DII vs. V5:** compare se as duas derivações apresentam padrão de separabilidade similar ou se uma delas é mais informativa para alguma classe específica. Isso antecipa decisões de seleção de derivações no Entregável 9.

---

## Seção 3 — Domínio da Frequência: PSD via Welch (12 Derivações)

### 3.1 Fundamentação

O método de **Welch** estima a Densidade Espectral de Potência (PSD) de forma estatisticamente robusta ao dividir o sinal em segmentos sobrepostos, calcular o periodograma de cada segmento e fazer a média. Comparado à FFT simples, o método de Welch reduz significativamente a variância da estimativa espectral — especialmente relevante para sinais biológicos, que são não-estacionários e contêm ruído aleatório.

Os parâmetros utilizados:
- **`nperseg = 256` (2,56 s):** janela longa o suficiente para boa resolução em frequência (resolução de ~0,39 Hz), mas curta o suficiente para capturar variações temporais relevantes do ECG.
- **`noverlap = 128` (50%):** sobreposição padrão que equilibra o compromisso entre variância reduzida e suavização excessiva.
- **Janela de Hann:** evita o *spectral leakage* que ocorre quando o sinal não é periódico dentro da janela.

As bandas fisiológicas integradas são escolhidas com base na literatura de análise espectral de ECG:

| Banda | Faixa (Hz) | Ondas Associadas |
|---|---|---|
| PT Band (P e T) | 0,5 – 5 Hz | Onda P, onda T, drift lento |
| QRS Band | 5 – 25 Hz | Complexo QRS (ativação ventricular) |
| Total Power | 0,5 – 40 Hz | Energia espectral total do sinal útil |

Além da potência por banda, extraímos três **descritores de forma da PSD**, que caracterizam a distribuição espectral sem depender de limiares de banda arbitrários:

- **Frequência de Pico:** frequência com maior densidade de potência — indica onde está concentrada a energia dominante do sinal.
- **Frequência Mediana:** frequência que divide a potência total em duas metades iguais — menos sensível a picos isolados que a frequência de pico.
- **Centroide Espectral:** centro de massa do espectro ponderado pela PSD — integra informação de toda a distribuição espectral.

### 3.2 Implementação

In [ ]:
def extract_freq_domain(sig_12l, fs=FS):
    """
    Calcula PSD via Welch e extrai potência em bandas e descritores espectrais.

    Parâmetros:
    - sig_12l : array (1000, 12) — registro de 10s com 12 derivações
    - fs      : frequência de amostragem (Hz)

    Retorna:
    - feats : dicionário com as features espectrais
    """
    feats = {}
    for i, lead in enumerate(LEAD_NAMES):
        s = sig_12l[:, i].astype(np.float64)

        # Estimação da PSD via Welch
        # nperseg=256 → janela de 2,56s → resolução em freq de ~0,39 Hz
        f, psd = signal.welch(s, fs=fs, window='hann',
                               nperseg=256, noverlap=128)

        # Segurança: evita divisão por zero em sinais com potência nula
        total_pwr = np.trapz(psd[(f >= 0.5) & (f <= 40)],
                              f[(f >= 0.5) & (f <= 40)])
        if total_pwr == 0:
            total_pwr = np.finfo(float).eps

        # 1. Potência em bandas fisiológicas (integração trapezoidal)
        mask_pt  = (f >= 0.5) & (f <= 5.0)
        mask_qrs = (f >= 5.0) & (f <= 25.0)
        mask_tot = (f >= 0.5) & (f <= 40.0)

        feats[f'freq_pt_power_{lead}']    = np.trapz(psd[mask_pt],  f[mask_pt])
        feats[f'freq_qrs_power_{lead}']   = np.trapz(psd[mask_qrs], f[mask_qrs])
        feats[f'freq_total_power_{lead}'] = total_pwr

        # 2. Razão QRS/Total (feature relativa, mais robusta a escala absoluta)
        feats[f'freq_qrs_ratio_{lead}'] = (
            feats[f'freq_qrs_power_{lead}'] / total_pwr
        )

        # 3. Descritores de forma da PSD
        psd_nz = psd[mask_tot]
        f_nz   = f[mask_tot]

        feats[f'freq_peak_{lead}']     = float(f_nz[np.argmax(psd_nz)])
        feats[f'freq_median_{lead}']   = float(
            f_nz[np.where(np.cumsum(psd_nz) >= np.sum(psd_nz) / 2)[0][0]]
        )
        feats[f'freq_centroid_{lead}'] = float(
            np.sum(f_nz * psd_nz) / np.sum(psd_nz)
        )

    return feats


print("Processando Domínio da Frequência em paralelo...")
freq_feats_list = Parallel(n_jobs=-1)(
    delayed(extract_freq_domain)(sinais_10s[i])
    for i in tqdm(range(len(df_reg_ids)), desc='Frequência')
)

df_freq = pd.DataFrame(freq_feats_list, index=df_reg_ids.index)

del freq_feats_list
gc.collect()

print(f"\nFeatures espectrais extraídas: {df_freq.shape[1]} colunas")
display(df_freq.describe().T[['mean', 'std', 'min', 'max']].head(10))

### 3.3 Visualização: PSD Média por Classe Diagnóstica

O gráfico abaixo mostra a **PSD média estimada na derivação DII** para cada superclasse diagnóstica. Esta visualização é fundamental para validar a separabilidade espectral entre as classes antes da classificação.

Padrões esperados:
- **CD (Distúrbio de Condução):** bloqueios de ramo alargam o complexo QRS, aumentando energia nas frequências mais baixas do espectro QRS (em torno de 5-10 Hz).
- **HYP:** maior amplitude das deflexões eleva a potência total, especialmente nas bandas de maior energia da onda R.
- **STTC (Alteração de ST-T):** alterações na onda T modificam a razão entre a banda PT e a banda QRS.

[Inserir análise comparativa das curvas — verificar se as diferenças entre classes são clinicamente coerentes e estatisticamente expressivas.]

In [ ]:
# Cálculo das PSDs médias por classe para visualização
def get_psd_mean_per_class(sinais_arr, df_meta, classes_target, fs=FS, lead_idx=1):
    """
    Calcula a PSD média via Welch para um subconjunto de registros por classe.
    Lead_idx=1 corresponde à derivação DII.
    """
    results = {}
    for cls in classes_target:
        ids_cls = df_meta[
            df_meta['diagnostic_superclass'].apply(
                lambda x: cls in x if isinstance(x, list) else False
            )
        ].index[:200]  # limitamos a 200 por classe para velocidade

        psds = []
        for eid in ids_cls:
            pos = df_meta.index.get_loc(eid)
            s = sinais_arr[pos, :, lead_idx].astype(np.float64)
            f, p = signal.welch(s, fs=fs, window='hann', nperseg=256, noverlap=128)
            psds.append(p)

        if psds:
            results[cls] = (f, np.mean(np.array(psds), axis=0))

    return results

classes_target = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
psd_por_classe = get_psd_mean_per_class(sinais_10s, df_reg_ids, classes_target)

fig, ax = plt.subplots(figsize=(14, 6))
palette = sns.color_palette('Spectral', n_colors=len(classes_target))

for (cls, (f_arr, psd_mean)), cor in zip(psd_por_classe.items(), palette):
    ax.semilogy(f_arr, psd_mean, label=cls, color=cor, linewidth=2)

# Marcação das bandas fisiológicas
ax.axvspan(0.5, 5,  alpha=0.07, color='blue',  label='Banda PT (0.5–5 Hz)')
ax.axvspan(5,  25,  alpha=0.07, color='orange', label='Banda QRS (5–25 Hz)')
ax.set_xlim(0.5, 40)
ax.set_xlabel('Frequência (Hz)')
ax.set_ylabel('Densidade Espectral de Potência [mV²/Hz]')
ax.set_title('PSD Média por Superclasse Diagnóstica — Derivação DII (Método de Welch)', fontsize=13)
ax.legend(fontsize=10, ncol=2)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'freq_psd_media_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

**📝 Comentários — PSD Média por Classe Diagnóstica (Método de Welch, DII):**

> *Preencha esta seção após executar a célula acima. Sugestões do que analisar:*

- **Formato geral das curvas:** em registros normais (NORM), espera-se um pico de potência na faixa de 10–20 Hz (complexo QRS) com queda progressiva fora dessa região. Verifique se esse comportamento é observado e se o pico está localizado na frequência esperada.
- **CD (Distúrbio de Condução / Bloqueios de Ramo):** bloqueios alargam o QRS, redistribuindo energia para frequências mais baixas da banda QRS (em torno de 5–10 Hz). Analise se a curva de CD apresenta "ombro" ou pico deslocado para a esquerda em relação à NORM.
- **HYP:** maior amplitude das deflexões cardíacas eleva a potência total, principalmente na banda QRS. Compare a potência absoluta de HYP versus NORM — espera-se que a curva de HYP esteja sistematicamente acima das demais.
- **STTC (Alteração de ST-T):** alterações na onda T modificam o conteúdo espectral na faixa de 0,5–5 Hz (banda PT). Verifique se a curva de STTC apresenta maior potência relativa nessa faixa comparada a NORM.
- **Sobreposição entre classes:** identifique as regiões de frequência onde as curvas das diferentes classes mais se separam — essas são as regiões mais informativas para as features espectrais que serão usadas no Entregável 9.

---

## Seção 4 — Morfologia de Ondas e Variabilidade de Ritmo (HRV)

### 4.1 Fundamentação

Esta seção opera sobre os **batimentos segmentados** produzidos no Entregável 5 — janelas de 600 ms (60 amostras a 100 Hz) centradas no pico R de cada batimento detectado pelo algoritmo Pan-Tompkins.

A análise em dois sub-grupos:

#### A. Morfologia das Ondas Cardíacas

Cada batimento individual permite medir grandezas morfológicas que, sobre o sinal de 10 s inteiro, seriam difíceis de isolar com precisão:

- **Amplitude R:** valor de pico da onda R (máximo da janela centrada no complexo QRS). Reduzida em infarto transmural pela perda de massa miocárdica; elevada em hipertrofia ventricular.
- **Duração do QRS:** largura estimada do complexo QRS por cruzamento de limiar. Alargamento (> 120 ms) é característico de bloqueios de ramo.
- **Segmento ST:** amplitude média no trecho pós-QRS (aprox. 60-120 ms após o pico R, equivalente a amostras 36-46). Elevação > 1 mm indica lesão aguda; depressão indica isquemia subendocárdica.
- **Amplitude T:** pico da onda T na porção final da janela (amostras 40-60). Onda T invertida em derivações precordiais pode indicar isquemia ou sobrecarga ventricular.

Os valores são extraídos **por batimento** e depois **agregados ao nível do registro** via mediana (robusta a batimentos outliers) e desvio padrão (captura variabilidade batimento a batimento).

#### B. Métricas de Variabilidade de Ritmo Cardíaco (HRV — Domínio do Tempo)

A variabilidade do intervalo RR — extraída da série de intervalos entre picos R consecutivos — é um dos indicadores fisiológicos mais estudados em cardiologia. Reflete o balanço autonômico entre a atividade simpática (reduz a variabilidade) e parassimpática (aumenta a variabilidade).

Métricas temporais de HRV extraídas:

| Métrica | Fórmula | Interpretação Clínica |
|---|---|---|
| meanRR | $\overline{RR}$ | Frequência cardíaca média |
| SDRR | $\sigma_{RR}$ | Variabilidade global do ritmo |
| RMSSD | $\sqrt{\overline{(\Delta RR_i)^2}}$ | Variabilidade de curto prazo (tônus vagal) |
| CV_RR | $\sigma_{RR} / \overline{RR}$ | Variabilidade normalizada pela FC média |

> **Nota sobre RMSSD:** é calculado sobre as diferenças **consecutivas** entre intervalos RR, capturando flutuações de curto prazo. É o índice temporal de HRV com maior associação à atividade parassimpática e ao tônus vagal, sendo amplamente utilizado em análises de saúde cardiovascular e monitoramento de estresse.

### 4.2 Implementação — Extração por Batimento

In [ ]:
def extract_morph_per_beat(beat_12l):
    """
    Extrai features morfológicas de um único batimento (60 amostras = 600 ms a 100 Hz).

    As medidas são focadas nas derivações DII (índice 1) e V5 (índice 10),
    que são as derivações de referência clínica para morfologia cardíaca:
    - DII: plano frontal, paralela ao eixo elétrico médio normal (~60°)
    - V5: precordial lateral, sensível à atividade ventricular esquerda

    Parâmetros:
    - beat_12l : array (60, 12) com um único batimento

    Retorna:
    - feats : dicionário {nome_feature: valor}
    """
    feats = {}

    for l_idx, l_name in [(1, 'II'), (10, 'V5')]:
        s = beat_12l[:, l_idx].astype(np.float64)

        # --- Amplitude R (pico máximo da janela) ---
        feats[f'morph_r_amp_{l_name}'] = float(np.max(s))

        # --- Duração do QRS (estimação por cruzamento de limiar) ---
        # Limiar: 15% do pico R — captura início e fim da deflexão principal
        peak_idx = int(np.argmax(s))
        thresh   = s[peak_idx] * 0.15
        try:
            qrs_pts = np.where(s > thresh)[0]
            # Largura em milissegundos (cada amostra = 10 ms a 100 Hz)
            feats[f'morph_qrs_width_ms_{l_name}'] = float((qrs_pts[-1] - qrs_pts[0]) * 10)
        except (IndexError, ValueError):
            # Fallback: valor mediano típico para QRS normal (~80 ms)
            feats[f'morph_qrs_width_ms_{l_name}'] = 80.0

        # --- Segmento ST ---
        # Amostras 36-46: corresponde aproximadamente ao ponto J + 60-100 ms
        # Região onde alterações isquêmicas são clinicamente avaliadas
        feats[f'morph_st_amp_{l_name}'] = float(np.mean(s[36:46]))

        # --- Amplitude da Onda T ---
        # Buscamos o pico máximo na janela final do batimento (amostras 40-60)
        # onde a onda T tipicamente ocorre para FC < 100 bpm
        if len(s) > 40:
            feats[f'morph_t_amp_{l_name}'] = float(np.max(s[40:]))
        else:
            feats[f'morph_t_amp_{l_name}'] = np.nan

        # --- Simetria do QRS (comparação das semi-amplitudes) ---
        # Assimetria entre a inclinação ascendente (ativação) e descendente (recuperação)
        mid  = len(s) // 2
        feats[f'morph_qrs_asym_{l_name}'] = float(
            np.mean(np.abs(s[:mid])) / (np.mean(np.abs(s[mid:])) + 1e-8)
        )

    return feats


print("Processando morfologia individual dos batimentos em paralelo...")
morph_list = Parallel(n_jobs=-1)(
    delayed(extract_morph_per_beat)(batimentos[i])
    for i in tqdm(range(len(batimentos)), desc='Morfologia')
)

df_morph_all = pd.DataFrame(morph_list)
print(f"Morfologia individual: {df_morph_all.shape[0]} batimentos × {df_morph_all.shape[1]} features")

### 4.3 Implementação — Agregação ao Nível do Registro e HRV

In [ ]:
# Associar cada batimento ao seu ecg_id e ao intervalo RR correspondente
df_morph_meta = pd.concat(
    [df_beat_ids[['ecg_id', 'rr_interval_ms']].reset_index(drop=True),
     df_morph_all.reset_index(drop=True)],
    axis=1
)

# ── A. Métricas de HRV (domínio do tempo) ──────────────────────────────────
def rmssd(x):
    """
    Calcula RMSSD sobre uma série de intervalos RR.
    Requer ao menos 2 valores não-nulos para calcular diferenças consecutivas.
    """
    s = x.dropna()
    if len(s) < 2:
        return np.nan
    return float(np.sqrt(np.mean(np.diff(s.values) ** 2)))

def cv_rr(x):
    """
    Coeficiente de variação dos intervalos RR.
    Normaliza a variabilidade pela FC média, permitindo comparação entre
    registros com frequências cardíacas diferentes.
    """
    mu = x.mean()
    return float(x.std() / mu) if mu != 0 else np.nan

df_hrv = df_morph_meta.groupby('ecg_id')['rr_interval_ms'].agg(
    hrv_meanRR = 'mean',
    hrv_sdRR   = 'std',
    hrv_rmssd  = rmssd,
    hrv_cvRR   = cv_rr,
    hrv_n_beats = 'count',   # Número de batimentos detectados no registro
)

# ── B. Agregação morfológica (Mediana + STD por registro) ───────────────────
# Mediana: robusta a batimentos individuais com valores extremos (artefatos residuais)
# STD: captura variabilidade batimento a batimento (ex: alternância elétrica, arritmias)
morph_cols = [c for c in df_morph_all.columns]
agg_rules  = {col: ['median', 'std'] for col in morph_cols}

df_morph_agg = df_morph_meta.groupby('ecg_id').agg(agg_rules)
df_morph_agg.columns = ['_'.join(col).strip() for col in df_morph_agg.columns.values]

# ── C. Unificação HRV + Morfologia ─────────────────────────────────────────
df_beat_final = df_hrv.join(df_morph_agg, how='left')

del morph_list, df_morph_all, df_morph_meta
gc.collect()

print(f"Features de morfologia e HRV agregadas: {df_beat_final.shape}")
print(f"  → {len(df_hrv.columns)} features de HRV")
print(f"  → {len(df_morph_agg.columns)} features morfológicas (mediana + STD)")
display(df_beat_final.head(3))

### 4.4 Visualização: HRV e Morfologia por Classe Diagnóstica

Os gráficos a seguir examinam a separabilidade entre as classes diagnósticas para as principais features morfológicas e de HRV.

[Inserir análise após execução — verificar se o RMSSD diferencia adequadamente classes com distúrbio de condução (CD), que costumam apresentar padrões de HRV alterados, em relação ao grupo NORM. Analisar se a amplitude R mediana em V5 separa HYP dos demais grupos.]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

classes_target = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

# Preparar DataFrame com label para plot
df_beat_plot = df_beat_final.copy()
df_beat_plot['label'] = df_reg_ids['diagnostic_superclass'].apply(get_primary)
df_beat_plot = df_beat_plot[df_beat_plot['label'].isin(classes_target)]

# Plot 1: meanRR (FC média)
sns.boxplot(data=df_beat_plot, x='label', y='hrv_meanRR', order=classes_target,
            hue='label', palette='Spectral', legend=False, ax=axes[0])
axes[0].set_title('Intervalo RR Médio por Classe')
axes[0].set_xlabel('Superclasse')
axes[0].set_ylabel('meanRR (ms)')

# Plot 2: RMSSD (variabilidade de curto prazo / tônus vagal)
sns.boxplot(data=df_beat_plot, x='label', y='hrv_rmssd', order=classes_target,
            hue='label', palette='Spectral', legend=False, ax=axes[1])
axes[1].set_title('RMSSD por Classe (Tônus Vagal)')
axes[1].set_xlabel('Superclasse')
axes[1].set_ylabel('RMSSD (ms)')
axes[1].set_yscale('log')

# Plot 3: Amplitude R mediana em V5
if 'morph_r_amp_V5_median' in df_beat_plot.columns:
    sns.boxplot(data=df_beat_plot, x='label', y='morph_r_amp_V5_median',
                order=classes_target, hue='label',
                palette='Spectral', legend=False, ax=axes[2])
    axes[2].set_title('Amplitude R Mediana (V5)')
    axes[2].set_xlabel('Superclasse')
    axes[2].set_ylabel('Amplitude (mV)')

plt.suptitle('Features de HRV e Morfologia por Classe Diagnóstica', fontsize=14)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'morph_hrv_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

**📝 Comentários — HRV e Morfologia por Classe Diagnóstica:**

> *Preencha esta seção após executar a célula acima. Sugestões do que analisar:*

- **meanRR (Frequência Cardíaca Média):** comente a distribuição do intervalo RR médio entre as classes. Classes associadas a taquicardia (MI agudo, STTC por isquemia) tendem a ter meanRR menor (FC maior); bradiarritmias (alguns bloqueios em CD) podem elevar o meanRR. Verifique se a dispersão intraclasse é grande, o que reduziria o poder discriminativo dessa feature isolada.
- **RMSSD (Tônus Vagal):** o RMSSD é o principal indicador de atividade parassimpática na HRV. Espera-se que registros normais apresentem RMSSD intermediário, enquanto patologias que afetam o sistema de condução (CD) ou que envolvem disfunção autonômica (MI) podem alterar essa métrica. Analise se os IQRs se sobrepõem muito ou se há separação estatisticamente útil.
- **Amplitude R Mediana em V5:** verifique se HYP se destaca das demais classes com amplitude R elevada, consistente com critérios de Sokolow-Lyon (soma de S em V1 + R em V5 > 35 mm). Se a separação for clara, essa feature terá alta relevância diagnóstica para HYP.
- **Variabilidade intraclasse:** analise se os boxplots são simétricos ou se há assimetria expressiva (distribuições com cauda longa para cima ou para baixo). Caudas longas podem indicar necessidade de transformação logarítmica para normalização no Entregável 7.

---

## Seção 5 — Domínio Tempo-Frequência: Transformada Wavelet Discreta (DWT)

### 5.1 Fundamentação

A **Transformada Wavelet Discreta (DWT)** decompõe o sinal em subbandas de frequência através de um banco de filtros em cascata, preservando simultaneamente informação temporal e espectral. Diferentemente da FFT — que assume estacionaridade global — a DWT é naturalmente adaptada a sinais não-estacionários como o ECG.

#### Estrutura da Decomposição (4 Níveis, db4)

A decomposição opera aplicando iterativamente um filtro **passa-baixa** (L, que gera os coeficientes de aproximação A) e um filtro **passa-alta** (H, que gera os coeficientes de detalhe D), seguidos de subamostragem por 2:

```
Sinal original (100 Hz)
    ├── A4: coeficientes de aproximação nível 4  → 0–6,25 Hz  (onda P, T, drift)
    ├── D4: detalhe nível 4                      → 6,25–12,5 Hz (início do QRS)
    ├── D3: detalhe nível 3                      → 12,5–25 Hz  (pico do QRS)
    ├── D2: detalhe nível 2                      → 25–50 Hz    (ruído residual)
    └── D1: detalhe nível 1                      → 50–100 Hz   (ruído de alta freq.)
```

> **Escolha da família db4 (Daubechies de ordem 4):** a wavelet-mãe db4 apresenta boa localização temporal e formato compatível com as morfologias das ondas cardíacas — sua assimetria e suavidade refletem a natureza das deflexões do ECG melhor que wavelets simétricas como a Haar.

Para cada subbanda, extraímos:

- **Energia:** $E_k = \sum_{n} c_k[n]^2$ — quantidade de sinal presente naquela faixa de frequência. Subbandas com maior energia concentram as estruturas dominantes do sinal.
- **Entropia de Shannon:** $H_k = -\sum_n p[n] \log_2 p[n]$, onde $p[n]$ é estimada via histograma dos coeficientes. Mede a distribuição da energia dentro da subbanda — coeficientes concentrados (ex: pico QRS isolado) têm menor entropia que coeficientes distribuídos.

### 5.2 Implementação

In [ ]:
def extract_wavelet_features(sig_lead, level=4, wavelet='db4'):
    """
    Aplica DWT de `level` níveis com família `wavelet` e extrai energia
    e entropia de Shannon por subbanda.

    Parâmetros:
    - sig_lead : array 1D com o sinal de uma derivação
    - level    : profundidade da decomposição wavelet
    - wavelet  : família wavelet (padrão: 'db4')

    Retorna:
    - feats : dicionário {nome_feature: valor}
    """
    def shannon_entropy(coefs):
        """
        Estima a entropia de Shannon via histograma dos coeficientes.
        O número de bins é determinado automaticamente (regra de Sturges).
        """
        counts, _ = np.histogram(coefs, bins='auto')
        # Filtra bins vazios para evitar log(0)
        counts = counts[counts > 0]
        probs = counts / counts.sum()
        return float(entropy(probs, base=2))

    feats = {}
    # pywt.wavedec retorna [cA_n, cD_n, cD_n-1, ..., cD_1]
    coeffs = pywt.wavedec(sig_lead, wavelet=wavelet, level=level)

    # Nomeação das subbandas para rastreabilidade
    band_names = [f'A{level}'] + [f'D{level - i}' for i in range(level)]

    for band_name, c in zip(band_names, coeffs):
        energy  = float(np.sum(c ** 2))
        ent     = shannon_entropy(c)
        norm_e  = energy / (np.sum(c ** 2) + 1e-10)  # Energia relativa

        feats[f'wavelet_energy_{band_name}']      = energy
        feats[f'wavelet_entropy_{band_name}']     = ent
        feats[f'wavelet_rel_energy_{band_name}']  = norm_e

    # Razão de energia QRS/Total: D3+D4 (12,5-25 Hz + 6,25-12,5 Hz) / energia total
    e_qrs   = feats['wavelet_energy_D3'] + feats['wavelet_energy_D4']
    e_total = sum(feats[k] for k in feats if 'wavelet_energy_' in k and 'rel' not in k)
    feats['wavelet_qrs_ratio'] = float(e_qrs / (e_total + 1e-10))

    return feats


# Aplicação sobre a derivação DII (índice 1) de todos os registros
print("Processando Domínio Tempo-Frequência (DWT, db4, 4 níveis) em DII...")

wavelet_feats_list = Parallel(n_jobs=-1)(
    delayed(extract_wavelet_features)(sinais_10s[i, :, 1].astype(np.float64))
    for i in tqdm(range(len(df_reg_ids)), desc='Wavelet')
)

df_wavelet = pd.DataFrame(wavelet_feats_list, index=df_reg_ids.index)

del wavelet_feats_list
gc.collect()

print(f"\nFeatures Wavelet extraídas: {df_wavelet.shape[1]} colunas")
display(df_wavelet.describe().T[['mean', 'std', 'min', 'max']])

### 5.3 Visualização: Decomposição Wavelet de Exemplos por Classe

O painel abaixo mostra a decomposição DWT completa para um registro representativo de cada classe diagnóstica principal. A inspeção visual permite verificar se as subbandas capturam corretamente as estruturas morfológicas esperadas:

- **A4 + D4:** componentes lentas (onda T, P, drift) e início do QRS.
- **D3:** pico principal do complexo QRS.
- **D2–D1:** componentes de alta frequência (ruído residual, entalhes do QRS).

[Inserir análise visual — verificar se, por exemplo, a subbanda D3 de registros com bloqueio de ramo (CD) apresenta energia mais distribuída no tempo que em registros NORM, refletindo o alargamento do QRS.]

In [ ]:
def plot_dwt_decomposicao(sinal_1d, titulo, fs=FS, wavelet='db4', level=4, ax_list=None):
    """
    Plota o sinal original e cada subbanda da decomposição DWT.
    """
    coeffs = pywt.wavedec(sinal_1d, wavelet=wavelet, level=level)
    band_names = [f'A{level}'] + [f'D{level - i}' for i in range(level)]

    n_plots = level + 2  # sinal original + subbandas
    if ax_list is None:
        fig, axes = plt.subplots(n_plots, 1, figsize=(14, 2.2 * n_plots))
    else:
        axes = ax_list

    t = np.arange(len(sinal_1d)) / fs
    axes[0].plot(t, sinal_1d, color='black', lw=0.7)
    axes[0].set_title(f'{titulo} — Sinal DII Original', fontsize=9)
    axes[0].set_ylabel('mV')

    for j, (bname, c) in enumerate(zip(band_names, coeffs)):
        tc = np.linspace(0, len(sinal_1d) / fs, len(c))
        axes[j + 1].plot(tc, c, color='steelblue', lw=0.8)
        axes[j + 1].set_title(f'Subbanda {bname}', fontsize=9)
        axes[j + 1].set_ylabel('Amp.')

    axes[-1].set_xlabel('Tempo (s)')
    return axes


classes_exemplo = ['NORM', 'MI', 'CD', 'HYP', 'STTC']
nivel_decomp = 4
n_plots_por_exemplo = nivel_decomp + 2  # sinal + 5 subbandas

fig, all_axes = plt.subplots(
    n_plots_por_exemplo, len(classes_exemplo),
    figsize=(5 * len(classes_exemplo), 2 * n_plots_por_exemplo),
    sharex='col'
)

for col_idx, cls in enumerate(classes_exemplo):
    ids_cls = df_reg_ids[
        df_reg_ids['diagnostic_superclass'].apply(
            lambda x: cls in x if isinstance(x, list) else False
        )
    ].index

    if len(ids_cls) == 0:
        continue

    eid = ids_cls[0]
    pos = df_reg_ids.index.get_loc(eid)
    sinal = sinais_10s[pos, :, 1].astype(np.float64)

    col_axes = all_axes[:, col_idx] if len(classes_exemplo) > 1 else all_axes
    plot_dwt_decomposicao(sinal, cls, ax_list=col_axes)

    if col_idx == 0:
        for ax in col_axes:
            ax.set_ylabel(ax.get_title().replace(f'{cls} — ', ''), fontsize=8)
            ax.set_title('')

plt.suptitle('Decomposição DWT (db4, 4 níveis) — Derivação DII por Classe', fontsize=14)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'wavelet_decomposicao_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

**📝 Comentários — Decomposição Wavelet DWT (db4, 4 Níveis) por Classe:**

> *Preencha esta seção após executar a célula acima. Sugestões do que analisar:*

- **Subbanda D3 (12,5–25 Hz) — complexo QRS:** esta subbanda deve concentrar a maior parte da energia em todos os registros, correspondendo ao pico principal do QRS. Verifique se o coeficiente D3 apresenta um pico bem definido e localizado temporalmente de forma consistente entre as classes.
- **CD (Bloqueio de Ramo):** bloqueios alargam o QRS, o que deve se manifestar como um pulso mais largo nos coeficientes D3 e D4. Compare visualmente a largura dos picos nessas subbandas entre CD e NORM — a diferença pode ser sutil a 100 Hz mas deve ser observável.
- **Subbanda A4 (0–6,25 Hz) — ondas lentas:** esta subbanda captura as ondas P, T e o drift residual. Em registros com alteração de ST ou onda T proeminente (STTC, MI), espera-se variação mais expressiva nos coeficientes de A4. Comente se essa diferença é visível.
- **D1 e D2 (alta frequência):** coeficientes de detalhes nos níveis mais altos de frequência devem apresentar amplitude baixa em sinais bem limpos (Entregável 4 removeu ruído nessa faixa). Se houver picos expressivos em D1, pode indicar artefatos residuais no registro mostrado.
- **Consistência entre exemplos da mesma classe:** analise se os registros de uma mesma classe apresentam padrão de decomposição similar ou se há variabilidade intraclasse expressiva — isso indica quão homogênea é a classe em termos de morfologia de sinal.

---

## Seção 6 — Dinâmica Não-Linear: DFA, Higuchi e Poincaré

### 6.1 Fundamentação

As métricas não-lineares capturam aspectos da dinâmica cardíaca que os domínios anteriores — baseados em estatísticas lineares de amplitude e frequência — não conseguem representar: a **complexidade geométrica** do sinal e as **correlações de longo alcance** na série temporal.

#### A. Dimensão Fractal de Higuchi (FD)

O método de Higuchi estima a dimensão fractal $D_H$ de uma série temporal construindo curvas de comprimento $L(k)$ para diferentes escalas temporais $k$ e ajustando a relação de potência:

$$L(k) \propto k^{-D_H}$$

Em sinais de ECG, valores típicos de $D_H$ situam-se entre 1,0 e 2,0. Coração saudável com dinâmica complexa tende a apresentar $D_H$ próximo de 1,5. Perda de complexidade (patologias) frequentemente reduz $D_H$ em direção a 1,0 (sinal mais regular).

O parâmetro `kmax=10` controla o número máximo de escalas temporais avaliadas — valores entre 8 e 12 são adequados para registros de ECG a 100 Hz.

#### B. DFA — Análise de Flutuações Sem Tendência

O DFA quantifica as **auto-correlações de longo alcance** em uma série temporal não-estacionária. O expoente de escala $\alpha$ é estimado pela relação entre o tamanho da janela de análise e a amplitude de flutuação residual após remoção de tendências locais:

$$F(n) \propto n^\alpha$$

Interpretação clínica:
- $\alpha \approx 1{,}0$: dinâmica fractal saudável (tipo ruído $1/f$)
- $\alpha > 1{,}5$: correlações suaves, sinal mais "suave" (possível patologia)
- $\alpha < 0{,}5$: anti-correlações (arritmias graves, fibrilação ventricular)

#### C. Entropia de Amostra (SampEn)

A Entropia de Amostra mede a probabilidade condicional de que sequências de comprimento $m$ que são semelhantes (diferença $< r$) continuem similares quando estendidas a comprimento $m+1$. Valores elevados indicam maior imprevisibilidade/irregularidade.

Para a série RR: SampEn elevada pode indicar fibrilação atrial (ritmo totalmente irregular); SampEn baixa pode indicar arritmias mecânicas ou bloqueios de grau avançado.

Parâmetros: $m=2$, $r=0{,}2 \times \sigma_{RR}$ — convenção padrão da literatura de HRV.

#### D. Diagrama de Poincaré — SD1 e SD2

O Diagrama de Poincaré plota cada intervalo RR contra o intervalo RR seguinte $(RR_i, RR_{i+1})$, formando uma nuvem de pontos cuja geometria revela padrões de variabilidade:

- **SD1:** desvio padrão ao longo do eixo de identidade rotacionado 45° — captura variabilidade de **curto prazo** (batimento a batimento), associada ao tônus parassimpático.
- **SD2:** desvio padrão ao longo do eixo perpendicular — captura variabilidade de **longo prazo**, associada à regulação simpática e parassimpática combinada.

$$SD1 = \sqrt{\frac{1}{2} \text{Var}(\Delta RR)} \quad SD2 = \sqrt{2\sigma_{RR}^2 - \frac{1}{2}\text{Var}(\Delta RR)}$$

### 6.2 Implementação

In [ ]:
def extract_nonlinear_features(sig_lead_1d, rr_series_ms):
    """
    Calcula métricas não-lineares para o sinal de uma derivação e a série RR.

    Parâmetros:
    - sig_lead_1d  : array 1D do sinal (derivação DII do registro de 10s)
    - rr_series_ms : lista ou array com os intervalos RR em ms

    Retorna:
    - feats : dicionário {nome_feature: valor}
    """
    feats = {}
    s = sig_lead_1d.astype(np.float64)

    # ── A. Dimensão Fractal de Higuchi ─────────────────────────────────────
    # kmax=10: número máximo de escalas temporais avaliadas
    # Escolhido com base no compromisso entre estabilidade da estimativa
    # e custo computacional para sinais de 1000 amostras a 100 Hz
    try:
        feats['nonlin_higuchi_fd'] = float(ant.higuchi_fd(s, kmax=10))
    except Exception:
        feats['nonlin_higuchi_fd'] = np.nan

    # ── B. DFA (expoente de escala α) ─────────────────────────────────────
    # antropy.detrended_fluctuation retorna diretamente o expoente α
    # estimado sobre múltiplas escalas log-espaçadas
    try:
        feats['nonlin_dfa_alpha'] = float(ant.detrended_fluctuation(s))
    except Exception:
        feats['nonlin_dfa_alpha'] = np.nan

    # ── C, D. Métricas sobre a série RR (requer ≥ 8 batimentos válidos) ───
    rr = np.array(rr_series_ms, dtype=np.float64)
    rr = rr[~np.isnan(rr)]

    if len(rr) >= 8:
        # SampEn: m=2, r=0.2*σ_RR (convenção padrão de HRV não-linear)
        try:
            feats['nonlin_sampen_rr'] = float(
                ant.sample_entropy(rr, order=2, metric='chebyshev')
            )
        except Exception:
            feats['nonlin_sampen_rr'] = np.nan

        # Poincaré SD1, SD2
        delta_rr = np.diff(rr)
        sd1 = float(np.sqrt(0.5 * np.var(delta_rr)))
        sd2 = float(np.sqrt(max(0, 2 * np.var(rr) - 0.5 * np.var(delta_rr))))

        feats['nonlin_sd1']      = sd1
        feats['nonlin_sd2']      = sd2
        # Razão SD1/SD2: reflete balanço simpato-vagal
        feats['nonlin_sd1_sd2_ratio'] = float(sd1 / (sd2 + 1e-10))

    else:
        for k in ['nonlin_sampen_rr', 'nonlin_sd1', 'nonlin_sd2',
                  'nonlin_sd1_sd2_ratio']:
            feats[k] = np.nan

    return feats


# Pré-agrupamento das séries RR por ecg_id (mais eficiente que query dentro do loop)
rr_groups = (
    df_beat_ids
    .dropna(subset=['rr_interval_ms'])
    .groupby('ecg_id')['rr_interval_ms']
    .apply(list)
)

print("Processando Dinâmica Não-Linear (Higuchi FD + DFA + SampEn + Poincaré) em DII...")
print("(Este módulo é computacionalmente custoso — tempo estimado: 10-30 min dependendo do hardware)")

nonlin_feats_list = []
for i, eid in enumerate(tqdm(df_reg_ids.index, desc='Não-Linear')):
    rr_s  = rr_groups.get(eid, [])
    lead  = sinais_10s[i, :, 1].astype(np.float64)
    res   = extract_nonlinear_features(lead, rr_s)
    nonlin_feats_list.append(res)

df_nonlinear = pd.DataFrame(nonlin_feats_list, index=df_reg_ids.index)

del nonlin_feats_list
gc.collect()

print(f"\nFeatures não-lineares extraídas: {df_nonlinear.shape[1]} colunas")
display(df_nonlinear.describe().T[['mean', 'std', 'min', 'max']])

### 6.3 Visualização: Diagrama de Poincaré por Classe

O Diagrama de Poincaré abaixo mostra a nuvem de pontos $(RR_i, RR_{i+1})$ para uma amostra de registros de cada classe diagnóstica. A forma da nuvem — mais compacta ou mais dispersa, circular ou elíptica — revela o padrão de variabilidade de ritmo característico de cada condição.

[Inserir análise visual — verificar se CD apresenta nuvem diferente de NORM (padrões de ritmo irregular), se HYP mostra concentração em valores de RR menores (taquicardia de esforço), etc.]

In [ ]:
fig, axes = plt.subplots(1, len(classes_target), figsize=(5 * len(classes_target), 5),
                         sharey=True)

for ax, cls in zip(axes, classes_target):
    ids_cls = df_reg_ids[
        df_reg_ids['diagnostic_superclass'].apply(
            lambda x: cls in x if isinstance(x, list) else False
        )
    ].index[:50]  # Limitamos a 50 registros por classe para legibilidade

    rr_concat_x, rr_concat_y = [], []
    for eid in ids_cls:
        rr_s = np.array(rr_groups.get(eid, []), dtype=np.float64)
        rr_s = rr_s[~np.isnan(rr_s)]
        if len(rr_s) >= 2:
            rr_concat_x.extend(rr_s[:-1].tolist())
            rr_concat_y.extend(rr_s[1:].tolist())

    ax.scatter(rr_concat_x, rr_concat_y, alpha=0.2, s=8, color='steelblue')
    ax.plot([400, 1400], [400, 1400], 'r--', lw=0.8, label='Identidade')
    ax.set_title(cls, fontsize=12)
    ax.set_xlabel('RR$_i$ (ms)')
    if cls == classes_target[0]:
        ax.set_ylabel('RR$_{i+1}$ (ms)')

plt.suptitle('Diagrama de Poincaré por Superclasse Diagnóstica', fontsize=14)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'nonlinear_poincare_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

**📝 Comentários — Diagrama de Poincaré por Superclasse Diagnóstica:**

> *Preencha esta seção após executar a célula acima. Sugestões do que analisar:*

- **NORM — elipse bem definida:** em corações saudáveis com regulação autonômica íntegra, a nuvem de pontos do Poincaré tende a formar uma elipse ao longo da linha de identidade, com eixo menor (SD1) menor que o eixo maior (SD2). Verifique se esse padrão é observado e se a nuvem é razoavelmente compacta.
- **CD (Distúrbio de Condução):** bloqueios atrioventriculares de grau avançado ou fibrilação atrial (frequentemente classificada em CD) produzem nuvens de Poincaré muito dispersas, sem forma elíptica clara. Analise se a nuvem de CD é visualmente mais espalhada que a de NORM.
- **MI e STTC:** infarto e alterações de ST podem cursar com disfunção autonômica que reduz a variabilidade geral — nuvens mais compactas e próximas da linha de identidade. Discuta se essa compactação é observada.
- **HYP:** hipertrofia ventricular muitas vezes não altera dramaticamente a variabilidade de ritmo, exceto quando associada a arritmias ventriculares. Comente se a nuvem de HYP é similar à de NORM ou apresenta diferenças observáveis.
- **Implicações para SD1 e SD2:** a partir da forma visual da nuvem, é possível antecipar se SD1 e SD2 serão features com boa separabilidade entre classes. Uma nuvem mais "redonda" indica SD1 ≈ SD2 (balanço simpato-vagal equilibrado); uma nuvem muito elíptica indica dominância simpática (SD1 << SD2).

---

## Seção 7 — Consolidação e Geração do Dataset Final (features_raw)

### 7.1 Unificação de todos os domínios

Reunimos aqui todos os DataFrames gerados nas seções anteriores em um único DataFrame indexado por `ecg_id`. O formato de saída é **Parquet**, preferível ao CSV neste caso por:

- Armazenamento mais eficiente para DataFrames com muitas colunas (> 200) e tipos mistos.
- Preservação dos tipos de dados (`float32`/`float64`) sem arredondamento por representação textual.
- Leitura seletiva de colunas sem carregar o arquivo completo em memória — útil para as etapas de Engenharia de Features e Seleção de Atributos.

Uma cópia em CSV é também salva para inspeção rápida e compatibilidade com ferramentas externas.

In [ ]:
# Unificação: join lateral de todos os sub-DataFrames pelo índice ecg_id
# O DataFrame base é df_reg_ids (metadados), ao qual agregamos as features
df_features_raw = df_reg_ids.join(
    [df_time, df_freq, df_beat_final, df_wavelet, df_nonlinear],
    how='left'
)

n_meta  = len(df_reg_ids.columns)
n_feats = len(df_features_raw.columns) - n_meta

print("═" * 55)
print("  SÍNTESE DA EXTRAÇÃO DE ATRIBUTOS — ENTREGÁVEL 6")
print("═" * 55)
display(Markdown(f"""
| Grupo de Features | Nº de Atributos |
|---|---|
| Domínio do Tempo (12 deriv.) | {len(df_time.columns)} |
| Domínio da Frequência (12 deriv.) | {len(df_freq.columns)} |
| Morfologia + HRV (batimento → registro) | {len(df_beat_final.columns)} |
| Wavelet DWT (db4, 4 níveis, DII) | {len(df_wavelet.columns)} |
| Dinâmica Não-Linear (DII + série RR) | {len(df_nonlinear.columns)} |
| **TOTAL DE ATRIBUTOS** | **{n_feats}** |
| Registros no dataset | {len(df_features_raw)} |
"""))

# ── Salvamento ────────────────────────────────────────────────────────────────
path_parquet = OUT_DIR / 'features_raw.parquet'
path_csv     = OUT_DIR / 'features_raw_sample.csv'

df_features_raw.to_parquet(str(path_parquet), index=True)
df_features_raw.to_csv(str(path_csv),     index=True)

print(f"\nArtefatos gerados:")
print(f"  Parquet : {path_parquet.resolve()}")
print(f"  CSV     : {path_csv.resolve()}")

---

## Seção 8 — Verificação de Sanidade do Dataset (Sanity Check)

Antes de encerrar este entregável, realizamos uma série de verificações quantitativas e qualitativas para garantir que o dataset gerado está íntegro e pronto para ser utilizado na etapa de Engenharia de Features.

### 8.1 Completude e Valores Ausentes

In [ ]:
# ── Completude geral ─────────────────────────────────────────────────────────
n_total  = df_features_raw.shape[0] * df_features_raw.shape[1]
n_nulls  = df_features_raw.isnull().sum().sum()
pct_null = 100 * n_nulls / n_total

print(f"Dimensão total do dataset  : {df_features_raw.shape}")
print(f"Valores ausentes totais    : {int(n_nulls)} ({pct_null:.2f}% das células)")

# Identificar colunas com maior proporção de NaNs
null_por_col = df_features_raw.isnull().mean().sort_values(ascending=False)
cols_com_null = null_por_col[null_por_col > 0]

if len(cols_com_null) > 0:
    print(f"\nColunas com valores ausentes ({len(cols_com_null)} no total):")
    display(cols_com_null.reset_index().rename(
        columns={'index': 'Feature', 0: 'Proporção de NaN'}
    ).head(20))
    print("\nAviso: valores ausentes concentram-se nas features não-lineares da série RR,")
    print("esperados para registros com menos de 8 batimentos detectados.")
    print("Serão tratados por imputação ou exclusão no Entregável 7.")
else:
    print("\nNenhum valor ausente detectado — dataset completo.")

In [ ]:
# ── Verificação de ranges esperados ──────────────────────────────────────────
print("Verificação de ranges para features-chave:\n")

checks = {
    'time_rms_II'      : (0.0, 5.0),    # RMS em mV (ECG típico < 3 mV)
    'freq_qrs_ratio_II': (0.0, 1.0),    # Razão entre 0 e 1
    'hrv_meanRR'       : (300, 2000),   # RR médio em ms (FC 30-200 bpm)
    'hrv_rmssd'        : (0.0, 500),    # RMSSD em ms
    'nonlin_higuchi_fd': (1.0, 2.0),    # Dimensão fractal (definição matemática)
    'nonlin_dfa_alpha' : (0.0, 2.5),    # Expoente DFA (limites práticos)
}

status_global = True
for feat, (vmin, vmax) in checks.items():
    if feat not in df_features_raw.columns:
        print(f"  [SKIP]  {feat} — coluna não encontrada")
        continue

    col      = df_features_raw[feat].dropna()
    fora     = ((col < vmin) | (col > vmax)).sum()
    pct_fora = 100 * fora / len(col) if len(col) > 0 else 0
    status   = "✓ OK" if pct_fora < 2 else "⚠ REVISAR"
    if pct_fora >= 2:
        status_global = False

    print(f"  [{status}]  {feat:35s}  range=({col.min():.3f}, {col.max():.3f})  "
          f"fora do esperado: {fora} ({pct_fora:.1f}%)")

print()
if status_global:
    print("Todas as verificações de range passaram dentro da tolerância de 2%.")
else:
    print("Atenção: algumas features apresentaram valores fora do range esperado.")
    print("Verificar se são outliers residuais (a serem tratados no Entregável 7).")

### 8.2 Tabela de Features Extraídas (Inventário Completo)

A célula abaixo gera e salva a tabela de inventário de features — lista completa com nome, domínio, derivação e interpretação clínica básica.

In [ ]:
def build_feature_catalog(df_feats, df_meta_cols):
    """
    Gera um DataFrame com inventário das features extraídas,
    classificando-as por domínio e fornecendo interpretação básica.
    """
    feat_cols = [c for c in df_feats.columns if c not in df_meta_cols]

    domain_map = {
        'time_'    : 'Tempo',
        'freq_'    : 'Frequência',
        'hrv_'     : 'HRV (Tempo)',
        'morph_'   : 'Morfologia',
        'wavelet_' : 'Tempo-Frequência (Wavelet)',
        'nonlin_'  : 'Não-Linear',
    }

    rows = []
    for col in feat_cols:
        domain = 'Outro'
        for prefix, d in domain_map.items():
            if col.startswith(prefix):
                domain = d
                break

        nan_pct = float(df_feats[col].isnull().mean() * 100)
        rows.append({
            'Feature'         : col,
            'Domínio'         : domain,
            'Média'           : round(float(df_feats[col].mean()), 4),
            'Desvio Padrão'   : round(float(df_feats[col].std()),  4),
            '% NaN'           : round(nan_pct, 2),
        })

    return pd.DataFrame(rows)


meta_cols = list(df_reg_ids.columns)
df_catalog = build_feature_catalog(df_features_raw, meta_cols)

# Salvamento do catálogo
catalog_path = OUT_DIR / 'feature_catalog.csv'
df_catalog.to_csv(str(catalog_path), index=False)

print(f"Catálogo salvo em: {catalog_path.resolve()}")
print(f"Total de features catalogadas: {len(df_catalog)}")
print()

# Resumo por domínio
display(df_catalog.groupby('Domínio').size()
        .reset_index(name='Nº de Features')
        .sort_values('Nº de Features', ascending=False))

### 8.3 Visualização Final: Mapa de Calor de Correlações entre Features-Chave

O heatmap a seguir mostra a matriz de correlação de Pearson entre as features mais representativas de cada domínio. Correlações muito altas (|r| > 0,90) entre features de domínios diferentes podem indicar redundância, que será tratada formalmente na etapa de Seleção de Atributos (Entregável 9).

[Inserir análise do heatmap — verificar pares com correlação elevada (ex: RMS e MAV são naturalmente correlacionados), correlações esperadas (SD1 e RMSSD são matematicamente equivalentes), e correlações inesperadas que podem revelar dependências não-óbvias entre domínios.]

In [ ]:
# Seleção de features representativas de cada domínio para o heatmap
cols_heatmap = []
for prefix in ['time_rms_II', 'time_zcr_II', 'time_skew_II',
                'freq_qrs_power_II', 'freq_qrs_ratio_II', 'freq_centroid_II',
                'hrv_meanRR', 'hrv_rmssd', 'hrv_sdRR',
                'morph_r_amp_II_median', 'morph_st_amp_II_median',
                'morph_qrs_width_ms_II_median',
                'wavelet_energy_D3', 'wavelet_energy_A4', 'wavelet_qrs_ratio',
                'nonlin_higuchi_fd', 'nonlin_dfa_alpha',
                'nonlin_sd1', 'nonlin_sd2', 'nonlin_sampen_rr']:
    if prefix in df_features_raw.columns:
        cols_heatmap.append(prefix)

df_corr = df_features_raw[cols_heatmap].corr(method='pearson')

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(df_corr, dtype=bool), k=1)

# Rótulos simplificados para legibilidade
labels = [c.replace('_II', '').replace('_median', '_med') for c in cols_heatmap]

sns.heatmap(
    df_corr,
    mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdBu_r', vmin=-1, vmax=1,
    xticklabels=labels, yticklabels=labels,
    linewidths=0.3, ax=ax, square=True
)
ax.set_title('Matriz de Correlação de Pearson — Features Representativas por Domínio',
             fontsize=13, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'heatmap_correlacao_features_chave.png', dpi=150, bbox_inches='tight')
plt.show()

**📝 Comentários — Matriz de Correlação entre Features Representativas:**

> *Preencha esta seção após executar a célula acima. Sugestões do que analisar:*

- **Redundâncias esperadas e verificadas:** comente se as correlações matematicamente esperadas aparecem no heatmap — por exemplo, RMS e MAV são medidas de energia com correlação alta esperada (|r| > 0,90); SD1 e RMSSD são algebricamente equivalentes ($SD1 = RMSSD / \sqrt{2}$) e devem apresentar correlação próxima de 1,0. Verifique se esses pares aparecem claramente no heatmap.
- **Redundâncias não-triviais entre domínios distintos:** identifique pares de features de domínios diferentes com correlação alta (|r| > 0,80). Por exemplo, se `wavelet_energy_D3` e `freq_qrs_power_II` apresentam correlação elevada, isso indica que ambas capturam essencialmente a mesma informação (energia do QRS), e uma delas pode ser eliminada na Seleção de Atributos (Entregável 9).
- **Features com baixa correlação com todas as demais:** features com correlações baixas com todos os outros atributos (|r| < 0,30) são potencialmente as mais informativas para o classificador, pois adicionam informação independente. Identifique quais grupos de features apresentam esse perfil.
- **Correlações negativas relevantes:** comente correlações negativas expressivas — por exemplo, `nonlin_higuchi_fd` (complexidade do sinal) pode ter correlação negativa com `hrv_meanRR` (FC elevada tende a reduzir a complexidade relativa do sinal). Discuta o significado fisiológico dessas relações.
- **Implicações para o Entregável 9:** com base nessa análise preliminar, indique quais pares de features são candidatos óbvios à remoção por redundância, antes mesmo da aplicação formal dos métodos de seleção.

---

## Seção 9 — Síntese e Conexão com o Entregável 7

### 9.1 O que foi feito neste entregável

Neste entregável, realizamos a transformação completa dos sinais de ECG brutos em um **dataset estruturado de features numéricas**, cobrindo quatro domínios complementares de análise:

- **Domínio do Tempo:** estatísticas descritivas de amplitude sobre as 12 derivações, capturando distribuição de energia e forma da distribuição de amplitudes.
- **Domínio da Frequência:** análise espectral via método de Welch, com integração de potência em bandas fisiológicas (PT, QRS) e descritores de forma da PSD.
- **Morfologia e HRV:** extração por batimento individual (via segmentação do Entregável 5) e agregação para o nível do registro, capturando amplitudes de ondas, durações de segmentos e variabilidade do ritmo cardíaco.
- **Domínio Tempo-Frequência:** decomposição Wavelet com família db4 em 4 níveis, extraindo energia e entropia por subbanda.
- **Dinâmica Não-Linear:** dimensão fractal de Higuchi, expoente DFA e métricas do Diagrama de Poincaré, capturando a complexidade intrínseca do sinal e das séries RR.

### 9.2 Limitações e decisões metodológicas relevantes

- **Restrição de domínios complexos à derivação DII:** para Wavelet, DFA, Higuchi e SampEn, o cálculo foi restrito à derivação DII por custo computacional. Trabalhos futuros podem estender esses cálculos para múltiplas derivações, potencialmente aumentando a discriminabilidade entre classes.

- **Valores ausentes nas features não-lineares da série RR:** registros com menos de 8 batimentos detectados (seja por sinal curto, arritmia severa ou falhas no detector Pan-Tompkins) resultam em NaN nessas features. Essa limitação é esperada e documentada no catálogo de features.

- **Sem normalização nesta etapa:** as features estão em suas escalas originais (mV, ms, Hz, adimensional). A normalização robusta será realizada no **Entregável 7 (Engenharia de Features)**, que também explorará features de segunda ordem (razões entre bandas, diferenças temporais) e tratará os valores ausentes.

### 9.3 Próximos passos — Entregável 7

O dataset `features_raw.parquet` gerado aqui constitui a entrada do **Entregável 7 (Engenharia de Features)**, que realizará:

1. **Normalização robusta** (por escalonamento baseado em mediana/IQR do conjunto de treino), evitando *data leakage* nos conjuntos de validação e teste.
2. **Criação de features derivadas** — razões entre bandas espectrais (ex: QRS/PT), deltas morfológicos (variação de amplitude R entre derivações), e combinações de métricas de HRV.
3. **Imputação dos valores ausentes** nas features não-lineares — por mediana do conjunto de treino ou por regressão baseada em features temporais correlacionadas.
4. **Análise de redundância** entre features dos diferentes domínios — preparando o dataset para a Seleção de Atributos do Entregável 9.

In [ ]:
# Verificação final consolidada
print("═" * 55)
print("   VERIFICAÇÃO FINAL — ENTREGÁVEL 6")
print("═" * 55)
print(f"  Dimensão do dataset final : {df_features_raw.shape}")
print(f"  Total de features         : {n_feats}")
print(f"  Valores ausentes totais   : {int(df_features_raw.isnull().sum().sum())}")
print(f"  Arquivos gerados:")
print(f"    - features_raw.parquet")
print(f"    - features_raw_sample.csv")
print(f"    - feature_catalog.csv")
print(f"    - {len(list(FIGS_DIR.glob('*.png')))} figuras em {FIGS_DIR}")
print()
print("  Dataset pronto para o Entregável 7 — Engenharia de Features.")
print("═" * 55)